# 01 · Baseline (untrained gemma-3-1b-it)
Generates answers for the 350 test pairs with the **untrained** model, using exactly the prompt and decoding settings the fine-tuned model will use.
Output: `eval/baseline_untrained/predictions_test.csv` in Drive.

Runtime: GPU (L4 or A100 preferred, T4 works). About 3 to 5 minutes.

In [ ]:
# ---- 1. Settings (edit once) ----
REPO_URL   = "https://github.com/O-sama2004/gemma-tr-temporal-inference.git"
DRIVE_ROOT = "/content/drive/MyDrive/gemma-tr-temporal-inference"
DATA_VERSION = "v1"
DATA = "/content/data"          # local copy (Drive I/O is slow)

In [ ]:
# ---- 2. Mount Drive, get the code ----
import os, subprocess, shutil
from google.colab import drive, userdata
drive.mount("/content/drive")

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

if not os.path.exists("/content/repo"):
    tok = secret("GITHUB_TOKEN")             # only needed if the repo is private
    url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
    subprocess.run(["git", "clone", "-q", url, "/content/repo"], check=True)
os.chdir("/content/repo")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# ---- 3. Pinned libraries (torch comes with Colab) ----
# If a later cell fails with an import error: Runtime > Restart session, then run cells 1, 2, 4, 5 again (skip this one).
!pip install -q -r requirements.txt

In [ ]:
# ---- 4. Hugging Face access ----
# One-time: accept the Gemma licence at https://huggingface.co/google/gemma-3-1b-it
# and add a read token as a Colab secret named HF_TOKEN (key icon on the left).
assert secret("HF_TOKEN"), "add the HF_TOKEN secret and allow this notebook to use it"
os.environ["HF_TOKEN"] = secret("HF_TOKEN")

In [ ]:
# ---- 5. Copy the frozen dataset to local disk and verify it ----
shutil.copytree(f"{DRIVE_ROOT}/data/{DATA_VERSION}", DATA, dirs_exist_ok=True)
!python src/verify_data.py {DATA}

In [ ]:
# ---- 6. GPU check ----
import torch
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0), "| dtype:", "bf16" if torch.cuda.is_bf16_supported() else "fp32 (T4: slower, still correct)")

## 7. Smoke test (8 items)
Check that the answers look like Turkish sentences before the full run.

In [ ]:
!python src/infer.py --data_dir {DATA} --split test --system baseline --limit 8 --out /content/smoke/pred_test.csv

## 8. Full test set

In [ ]:
OUT = f"{DRIVE_ROOT}/eval/baseline_untrained/predictions_test.csv"
!python src/infer.py --data_dir {DATA} --split test --system baseline --out "{OUT}"

In [ ]:
import pandas as pd
p = pd.read_csv(OUT, keep_default_na=False)
print(len(p), "predictions |", (p.prediction == "").sum(), "empty")
p.sample(10, random_state=0)[["question", "answer", "prediction"]]